In this session we are going to study the POS-tagging via Hidden Markov Model and Viterbi algorithm. It includes:

1. Black-box baselines (NLTK, spaCy)
2. HMM training from counts (toy corpus)
3. Transition/Emission tables (readable)
4. Viterbi decoding
5. Viterbi DP table demo on a reduced tag set


## Modules

In [ ]:
!pip -q install spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 74.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import math
from collections import defaultdict, Counter

# ------------------------------------------------------------
# 0. Quick baselines (black-box taggers): NLTK + spaCy
# ------------------------------------------------------------

print("\n=== Baselines (black-box taggers) ===")

import nltk
from nltk import word_tokenize, pos_tag
nltk.download("punkt_tab")
nltk.download("averaged_perceptron_tagger_eng")

# NLP - pipeline
import spacy
from spacy import displacy
nlp = spacy.load("en_core_web_sm")


=== Baselines (black-box taggers) ===


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


## Baselines

In [ ]:
s0 = "The quick brown fox jumps over the lazy dog."
tokens0 = word_tokenize(s0)
tags0 = pos_tag(tokens0)

print("\nNLTK:")
for w, t in tags0:
    print(f"{w:>10}  {t}")


NLTK:
       The  DT
     quick  JJ
     brown  NN
       fox  NN
     jumps  VBZ
      over  IN
       the  DT
      lazy  JJ
       dog  NN
         .  .


Now with SpaCy. First let's see which are the common components of the nlp pipeline

In [ ]:
print("Pipeline components:", nlp.pipe_names)

Pipeline components: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


In [ ]:
print("\nspaCy:")
doc0 = nlp(s0)

for w, p, g in zip(
    [t.text for t in doc0],
    [t.pos_ for t in doc0],
    [t.tag_ for t in doc0]
):
    print(f"{w:>10}  {p}  {g}")


spaCy:
       The  DET  DT
     quick  ADJ  JJ
     brown  ADJ  JJ
       fox  NOUN  NN
     jumps  VERB  VBZ
      over  ADP  IN
       the  DET  DT
      lazy  ADJ  JJ
       dog  NOUN  NN
         .  PUNCT  .


Or a slightly more detailed version as

In [ ]:
for token1 in doc0:
  print(token1, "|", token1.pos_,"|", spacy.explain(token1.pos_),"|",token1.tag_, spacy.explain(token1.tag_))

The | DET | determiner | DT determiner
quick | ADJ | adjective | JJ adjective (English), other noun-modifier (Chinese)
brown | ADJ | adjective | JJ adjective (English), other noun-modifier (Chinese)
fox | NOUN | noun | NN noun, singular or mass
jumps | VERB | verb | VBZ verb, 3rd person singular present
over | ADP | adposition | IN conjunction, subordinating or preposition
the | DET | determiner | DT determiner
lazy | ADJ | adjective | JJ adjective (English), other noun-modifier (Chinese)
dog | NOUN | noun | NN noun, singular or mass
. | PUNCT | punctuation | . punctuation mark, sentence closer


Let's change the sentence to have some ambiguous words

In [ ]:
doc1 = nlp("Apple is planning to buy Indian startup for $1 billion")
for token1 in doc1:
  print(token1, "|", token1.pos_,"|", spacy.explain(token1.pos_),"|",token1.tag_, spacy.explain(token1.tag_))

Apple | PROPN | proper noun | NNP noun, proper singular
is | AUX | auxiliary | VBZ verb, 3rd person singular present
planning | VERB | verb | VBG verb, gerund or present participle
to | PART | particle | TO infinitival "to"
buy | VERB | verb | VB verb, base form
Indian | ADJ | adjective | JJ adjective (English), other noun-modifier (Chinese)
startup | NOUN | noun | NN noun, singular or mass
for | ADP | adposition | IN conjunction, subordinating or preposition
$ | SYM | symbol | $ symbol, currency
1 | NUM | numeral | CD cardinal number
billion | NUM | numeral | CD cardinal number


In [ ]:
doc = nlp("""Apple Inc. is an American multinational technology company headquartered in Cupertino, California. It was founded by Steve Jobs, Steve Wozniak, and Ronald Wayne in April 1976. Apple designs, manufactures, and sells consumer electronics, computer software, and online services. Its best-known hardware products include the iPhone, iPad, Mac, and Apple Watch.""")
for ent in doc.ents:
    print(ent.text, " | ", ent.label_, " | ", spacy.explain(ent.label_))

Apple Inc.  |  ORG  |  Companies, agencies, institutions, etc.
American  |  NORP  |  Nationalities or religious or political groups
Cupertino  |  GPE  |  Countries, cities, states
California  |  GPE  |  Countries, cities, states
Steve Jobs  |  PERSON  |  People, including fictional
Steve Wozniak  |  PERSON  |  People, including fictional
Ronald Wayne  |  PERSON  |  People, including fictional
April 1976  |  DATE  |  Absolute or relative dates or periods
Apple  |  ORG  |  Companies, agencies, institutions, etc.
iPhone  |  ORG  |  Companies, agencies, institutions, etc.
iPad  |  ORG  |  Companies, agencies, institutions, etc.
Mac  |  PERSON  |  People, including fictional
Apple Watch  |  ORG  |  Companies, agencies, institutions, etc.


Note what happened with "Apple"? (This is in advance for the Named Entity Recognition session, but look how the entities are found)

In [ ]:
displacy.render(doc, style="ent")

### Activity 1:

Create 2 minimal pairs where the same word changes POS.
Words: can, book, light, bank.
Explain which context feature changes the tag.

## Tiny HMM from counts

In [ ]:
print("\n=== Tiny HMM training from counts ===")

train_sents = [
    [("the","DET"), ("quick","ADJ"), ("brown","ADJ"), ("fox","NOUN"), ("jumps","VERB"), ("over","ADP"), ("the","DET"), ("lazy","ADJ"), ("dog","NOUN"), (".","PUNCT")],
    [("the","DET"), ("dog","NOUN"), ("runs","VERB"), (".","PUNCT")],
    [("the","DET"), ("cat","NOUN"), ("runs","VERB"), (".","PUNCT")],
    [("a","DET"), ("quick","ADJ"), ("dog","NOUN"), ("runs","VERB"), (".","PUNCT")],
    [("I","PRON"), ("am","AUX"), ("learning","VERB"), ("nlp","NOUN"), (".","PUNCT")],
    [("they","PRON"), ("can","AUX"), ("fish","VERB"), (".","PUNCT")],
    [("they","PRON"), ("bought","VERB"), ("a","DET"), ("can","NOUN"), ("of","ADP"), ("fish","NOUN"), (".","PUNCT")],
]

START = "<s>"
END = "</s>"
UNK = "<unk>"

transition_counts = defaultdict(Counter)  # prev_tag -> Counter(next_tag)
emission_counts = defaultdict(Counter)    # tag -> Counter(word)
tag_counts = Counter()
vocab = set()

for sent in train_sents:
    prev = START
    transition_counts[START][sent[0][1]] += 1  # START -> first tag
    for word, tag in sent:
        vocab.add(word)
        emission_counts[tag][word] += 1
        tag_counts[tag] += 1
        if prev != START:
            transition_counts[prev][tag] += 1
        prev = tag
    transition_counts[prev][END] += 1

tags = sorted(tag_counts.keys())

for t in tags:
    emission_counts[t][UNK] += 0

print("Tags:", tags)
print("Vocab size:", len(vocab))


=== Tiny HMM training from counts ===
Tags: ['ADJ', 'ADP', 'AUX', 'DET', 'NOUN', 'PRON', 'PUNCT', 'VERB']
Vocab size: 21


Note that in the code above we have ensured that UNK is present in emission tables (count 0; smoothing handles it)

Now we can define the smoothed probability helpers (just a simple add-k smoothing hyperparameter)

In [ ]:
def log_prob_transition(prev_tag, next_tag, k=1.0):
    """log P(next_tag | prev_tag) with add-k smoothing."""
    next_space = tags + [END]
    num = transition_counts[prev_tag][next_tag] + k
    den = sum(transition_counts[prev_tag][t] for t in next_space) + k * len(next_space)
    return math.log(num / den)

def log_prob_emission(tag, word, k=1.0):
    """log P(word | tag) with add-k smoothing + UNK handling."""
    w = word if word in vocab else UNK
    num = emission_counts[tag][w] + k
    den = sum(emission_counts[tag][x] for x in vocab) + emission_counts[tag][UNK] + k * (len(vocab) + 1)
    return math.log(num / den)

def prob_from_log(logp):
    return math.exp(logp)

In [ ]:
def show_top_transitions(prev_tag, top_k=6, k=1.0):
    candidates = tags + [END]
    scored = [(nxt, prob_from_log(log_prob_transition(prev_tag, nxt, k=k))) for nxt in candidates]
    scored.sort(key=lambda x: x[1], reverse=True)
    print(f"\nTop transitions from {prev_tag}:")
    for nxt, p in scored[:top_k]:
        print(f"  {prev_tag:>6} -> {nxt:<6}  {p:.3f}")

def show_top_emissions(tag, top_k=8, k=1.0):
    # compute for a subset: seen words + UNK
    candidates = list(vocab) + [UNK]
    scored = [(w, prob_from_log(log_prob_emission(tag, w, k=k))) for w in candidates]
    scored.sort(key=lambda x: x[1], reverse=True)
    print(f"\nTop emissions for {tag} (P(word|{tag})):")
    for w, p in scored[:top_k]:
        w_disp = w if w != UNK else "<UNK>"
        print(f"  {tag:>6} -> {w_disp:<10} {p:.3f}")

Let's display a few key ones that connect to ambiguity and sequence structure

In [ ]:
print("\n=== Readable probability tables (selected) ===")

show_top_transitions("DET")
show_top_transitions("PRON")
show_top_transitions("AUX")
show_top_transitions("NOUN")

show_top_emissions("DET")
show_top_emissions("AUX")
show_top_emissions("NOUN")
show_top_emissions("VERB")


=== Readable probability tables (selected) ===

Top transitions from DET:
     DET -> ADJ     0.267
     DET -> NOUN    0.267
     DET -> ADP     0.067
     DET -> AUX     0.067
     DET -> DET     0.067
     DET -> PRON    0.067

Top transitions from PRON:
    PRON -> AUX     0.250
    PRON -> VERB    0.167
    PRON -> ADJ     0.083
    PRON -> ADP     0.083
    PRON -> DET     0.083
    PRON -> NOUN    0.083

Top transitions from AUX:
     AUX -> VERB    0.273
     AUX -> ADJ     0.091
     AUX -> ADP     0.091
     AUX -> AUX     0.091
     AUX -> DET     0.091
     AUX -> NOUN    0.091

Top transitions from NOUN:
    NOUN -> VERB    0.294
    NOUN -> PUNCT   0.235
    NOUN -> ADP     0.118
    NOUN -> ADJ     0.059
    NOUN -> AUX     0.059
    NOUN -> DET     0.059

Top emissions for DET (P(word|DET)):
     DET -> the        0.179
     DET -> a          0.107
     DET -> am         0.036
     DET -> I          0.036
     DET -> runs       0.036
     DET -> of         0.036
     D

### Activity 2:

Extend the training data with 2 sentences containing a new word and/or an ambiguous word ("can", "book"). Rerun the tables and observe the changes.


## Viterbi Decoding

Let's now define the Viterbi algorithm explicitely with HHM POS tagging to return the best tag and the best log-probability of the tag

In [ ]:
def viterbi_decode(words, k_trans=1.0, k_emit=1.0):
    dp = []
    back = []

    # init
    dp0, back0 = {}, {}
    for t in tags:
        dp0[t] = log_prob_transition(START, t, k=k_trans) + log_prob_emission(t, words[0], k=k_emit)
        back0[t] = START
    dp.append(dp0)
    back.append(back0)

    # recursion
    for i in range(1, len(words)):
        dpi, backi = {}, {}
        for t in tags:
            best_prev, best_score = None, -1e100
            for prev_t in tags:
                score = dp[i-1][prev_t] + log_prob_transition(prev_t, t, k=k_trans) + log_prob_emission(t, words[i], k=k_emit)
                if score > best_score:
                    best_score = score
                    best_prev = prev_t
            dpi[t] = best_score
            backi[t] = best_prev
        dp.append(dpi)
        back.append(backi)

    # terminate
    best_last, best_last_score = None, -1e100
    for t in tags:
        score = dp[-1][t] + log_prob_transition(t, END, k=k_trans)
        if score > best_last_score:
            best_last_score = score
            best_last = t

    # backtrack
    best_path = [best_last]
    for i in range(len(words)-1, 0, -1):
        best_path.append(back[i][best_path[-1]])
    best_path.reverse()

    return best_path, best_last_score

Let's use it

In [ ]:
print("\n=== Viterbi decoding examples ===")

tests = [
    "they can fish .".split(),
    "they bought a can of fish .".split(),
    "the quick dog runs .".split(),
    "the can rusts .".split(),  # rusts is UNK
]

for sent in tests:
    pred_tags, score = viterbi_decode(sent, k_trans=1.0, k_emit=1.0)
    print("\nSentence:", " ".join(sent))
    print("Viterbi:", list(zip(sent, pred_tags)))
    print("log-score:", round(score, 3))


=== Viterbi decoding examples ===

Sentence: they can fish .
Viterbi: [('they', 'PRON'), ('can', 'AUX'), ('fish', 'VERB'), ('.', 'PUNCT')]
log-score: -14.495

Sentence: they bought a can of fish .
Viterbi: [('they', 'PRON'), ('bought', 'VERB'), ('a', 'DET'), ('can', 'ADJ'), ('of', 'NOUN'), ('fish', 'VERB'), ('.', 'PUNCT')]
log-score: -28.487

Sentence: the quick dog runs .
Viterbi: [('the', 'DET'), ('quick', 'ADJ'), ('dog', 'NOUN'), ('runs', 'VERB'), ('.', 'PUNCT')]
log-score: -15.91

Sentence: the can rusts .
Viterbi: [('the', 'DET'), ('can', 'NOUN'), ('rusts', 'VERB'), ('.', 'PUNCT')]
log-score: -14.651


Let's interpret a few facts of the output:

* The first sentence is fine with the different tag predicitions right for the sentence itself: ('they', 'PRON'), ('can', 'AUX'), ('fish', 'VERB'), and crucially: PRON to AUX and AUX to VERB are plausible and frequent transitions

* The second sentence is wrong in the middle: the model predicts ('can', 'ADJ'), ('of', 'NOUN'), ('fish', 'VERB'), it is mainly because the corpus is extremely small and unbalanced, but shows how the prediction can fail due to few counts and add-k smoothing

* The third and fourth sentences are rightly predicted/interpreted: note that for the third there is no need for interpretation, but the fourth needs it due to "can" not be the "auxiliar verb" but the "noun".


The log-probs themselves cannot be interpreted across the sentences since they come from logs of multiplication of probabilities and hence the number of words affects the results. We should only compare different log-probabilities for the same sentence. How can we get different predicitions? In this simple model a few options are

 * Changing the smoothing: larger-k gives more uniform probabilities and then weaker influence of counts, small-k gives sharper influence of observed data
 * Modifying the training data: adding sentences will change the probability of some transitions
 * An update that we can do is compute the top-k predictions, not just the best one

### Activity 3:

Pick one sentence (e.g., "they can fish .").

1. Write 2 plausible tag sequences.
2. Explain Viterbi’s choice using transitions vs emissions.
3. Modify ONE training sentence so that prediction flips.
4. Add a new sentence to the corpus (make it significantly larger than the others) and then estimate different log-probs by using the first technique explained above. Interpret the results of the log-probs and what this implies with respect to your prediction.

## Viterbi Dynamic Prgramming Table

The code below determines the DP table of the Viterbi algorithm above. In the table we will present, each number represents the best log-probability of any tag sequence that ends at that word with that tag

In [ ]:
def viterbi_debug_table(words, allowed_tags, k_trans=1.0, k_emit=1.0):
    """
    Prints a compact Viterbi DP table:
    rows = tags, columns = word positions, values = best log-score so far.
    Also prints the chosen best tag at each step (local best), and final path.
    """
    # dp[i][t] and back[i][t]
    dp = []
    back = []

    # init
    dp0, back0 = {}, {}
    for t in allowed_tags:
        dp0[t] = log_prob_transition(START, t, k=k_trans) + log_prob_emission(t, words[0], k=k_emit)
        back0[t] = START
    dp.append(dp0)
    back.append(back0)

    # recursion
    for i in range(1, len(words)):
        dpi, backi = {}, {}
        for t in allowed_tags:
            best_prev, best_score = None, -1e100
            for prev_t in allowed_tags:
                score = dp[i-1][prev_t] + log_prob_transition(prev_t, t, k=k_trans) + log_prob_emission(t, words[i], k=k_emit)
                if score > best_score:
                    best_score = score
                    best_prev = prev_t
            dpi[t] = best_score
            backi[t] = best_prev
        dp.append(dpi)
        back.append(backi)

    # terminate
    best_last, best_last_score = None, -1e100
    for t in allowed_tags:
        score = dp[-1][t] + log_prob_transition(t, END, k=k_trans)
        if score > best_last_score:
            best_last_score = score
            best_last = t

    # backtrack
    best_path = [best_last]
    for i in range(len(words)-1, 0, -1):
        best_path.append(back[i][best_path[-1]])
    best_path.reverse()

    # print table header
    print("\nSentence:", " ".join(words))
    header = "tag\\pos".ljust(8) + "".join(w.center(12) for w in words)
    print(header)
    print("-" * len(header))

    # print scores
    for t in allowed_tags:
        row = t.ljust(8)
        for i in range(len(words)):
            row += f"{dp[i][t]:12.2f}"
        print(row)

    # show local best tag per position (not the final Viterbi path)
    local_best = []
    for i in range(len(words)):
        best_t = max(allowed_tags, key=lambda t: dp[i][t])
        local_best.append(best_t)
    print("\nLocal best per step:", list(zip(words, local_best)))
    print("Final Viterbi path :", list(zip(words, best_path)))
    print("Final log-score    :", round(best_last_score, 3))

and now let's see it at work

In [ ]:
print("\n=== Viterbi DP table demo (small tag set) ===")
sent_demo = "they can fish .".split()
allowed = ["PRON", "AUX", "NOUN", "VERB", "PUNCT"]
viterbi_debug_table(sent_demo, allowed_tags=allowed, k_trans=1.0, k_emit=1.0)


=== Viterbi DP table demo (small tag set) ===

Sentence: they can fish .
tag\pos     they        can         fish         .      
--------------------------------------------------------
PRON           -3.51       -9.21      -12.99      -17.34
AUX            -5.95       -7.38      -12.95      -17.30
NOUN           -6.17       -8.70      -12.48      -16.83
VERB           -6.14       -8.67      -11.35      -17.07
PUNCT          -6.14       -9.36      -13.14      -13.80

Local best per step: [('they', 'PRON'), ('can', 'AUX'), ('fish', 'VERB'), ('.', 'PUNCT')]
Final Viterbi path : [('they', 'PRON'), ('can', 'AUX'), ('fish', 'VERB'), ('.', 'PUNCT')]
Final log-score    : -14.495


For the first column the best score is -3.51 and therefore the best tag for the first word is PRON. This means that it is the most likely at that position since $P(PRON|START)$ and $P(they|PRON)$ are large compared to the other options.

In the second column the largest log-prob is -7.38 which implies that the best path up to "they can" ends in the tag AUX, what we are findind now are transition probabilities from PRON to any of the others and emssion probabilities of "can" given the tags.

The rest is the same.

### Activity 4:

Change allowed tags (add DET, ADP, ADJ) and see how the DP table changes Explain why adding tags can change the optimal path.
